# 📘 Введение: Базовые принципы трансформеров (от RNN к Self-Attention)

Модели на базе трансформеров радикально изменили ландшафт машинного обучения, особенно в области **обработки последовательной информации**: текст, аудиосигналы, временные ряды, ДНК-последовательности и даже видео.

Эта лекция познакомит нас с эволюцией архитектур от **рекуррентных нейросетей (RNN)** до **self-attention**, лежащего в основе современных моделей, таких как GPT, BERT, T5 и др.

---

## 🧭 Зачем это важно?

Когда-то RNN были основным инструментом в задачах, связанных с последовательностями: машинный перевод, генерация текста, анализ тональности. Но со временем стало ясно, что RNN:

- плохо масштабируются;
- не умеют эффективно "запоминать" информацию на длинных расстояниях;
- вычислительно медленны из-за своей последовательной природы.

Решением стал механизм **внимания (Attention)**, позволивший «смотреть» на все элементы входа одновременно — и **обрабатывать последовательности параллельно**.

---

## 🎯 Цели лекции

**К концу занятия вы должны:**

- Понимать устройство и механику RNN, а также их ключевые слабости.
- Осознавать, как и почему появился механизм Attention.
- Разобрать архитектуру Transformer и идею self-attention.
- Отличать трансформерные модели от сверточных (CNN), понимать области их применения.
- Уметь реализовать простейшую версию внимания вручную (векторно/матрично).

---

## 🗂️ Структура лекции

1. **Что такое последовательные данные и почему это сложно.**
   - Примеры: текст, речь, временные ряды.
   - Почему полносвязные и сверточные сети не подходят.

2. **Рекуррентные нейросети (RNN).**
   - Принцип работы.
   - Пример: генерация по синусоиде.
   - Проблемы: исчезающие/взрывающиеся градиенты.

3. **Улучшения: LSTM, GRU.**
   - Пример, чем отличаются от обычных RNN.
   - Почему всё ещё не идеально.

4. **Переход к Attention.**
   - Главная идея: смотреть не только назад.
   - Пример ручной реализации внимания на NumPy.
   - Визуализация внимания.

5. **Transformer: Attention Is All You Need (2017).**
   - Encoder / Decoder.
   - Позиционное кодирование.
   - Self-attention и Multi-head attention.

6. **Сравнение CNN и Transformer.**
   - Механика восприятия локального и глобального контекста.
   - Где трансформеры выигрывают, а где — нет.

7. **Заключение + связь с современными моделями (BERT, GPT, T5).**
   - Как развивается идея Attention дальше.

---

## 🛠️ Что мы будем делать в ноутбуке

- Визуализируем архитектуры RNN и Attention.
- Реализуем mini-RNN и attention на синтетических данных.
- Используем готовые модели HuggingFace Transformers для демонстрации перевода и генерации.
- Поработаем с визуализацией весов внимания.

# 📘 Введение: Байесовская природа Embedding и влияние размерности

В этом блоке лекции мы рассмотрим:

- Что такое embedding и как он работает
- Байесовскую интерпретацию embedding
- Как размерность embedding влияет на качество модели
- Простую симуляцию влияния размерности на точность классификации


# 📖 Теория: Что такое Embedding

Embedding — это способ преобразования категориальных/дискретных объектов в векторы фиксированной длины.
Пример: слово "cat" → [0.12, -1.03, 0.44, ..., 0.81]

Embedding обучается как часть нейросети и используется, чтобы кодировать семантику объектов в числовом виде.


# 🧠 Байесовская интерпретация Embedding

Можно рассматривать embedding как аппроксимацию распределения вероятностей на скрытом (латентном) пространстве.

Пример с Word2Vec:
P(context | word) ≈ softmax(w_i · c_j)

Где:
- w_i — embedding слова
- c_j — embedding слова-контекста

Таким образом, вектора можно трактовать как логарифмы вероятностей:
log P(context | word) ≈ w_i · c_j


# 📊 Почему размер embedding важен?

- Слишком маленькая размерность → underfitting, потеря информации
- Слишком большая → overfitting, много параметров, сложность модели

Обычно:
- для 10k слов достаточно 100–300
- для <1k ID — 16–64

Далее — демонстрация зависимости точности от размерности embedding.


In [ ]:
# 🔧 Установка (если нужно)
# !pip install torch matplotlib seaborn


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sns.set(style="whitegrid")


In [ ]:
# 📊 Генерация дискретных ID и классов
np.random.seed(42)
torch.manual_seed(42)

n_samples = 1000
n_ids = 50
n_classes = 4

X = np.random.randint(0, n_ids, size=(n_samples,))
y = np.random.randint(0, n_classes, size=(n_samples,))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


 Обучение модели с разной размерностью embedding

In [ ]:
# ⚙️ Обучение простой модели с embedding и линейным классификатором
def train_model(embedding_dim=8):
    model = nn.Sequential(
        nn.Embedding(n_ids, embedding_dim),
        nn.Flatten(),
        nn.Linear(embedding_dim, n_classes)
    )
    
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

    X_tensor = torch.tensor(X_train, dtype=torch.long)
    y_tensor = torch.tensor(y_train, dtype=torch.long)

    for epoch in range(100):
        optimizer.zero_grad()
        out = model(X_tensor)
        loss = loss_fn(out, y_tensor)
        loss.backward()
        optimizer.step()
    
    # Тестирование
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.long)
        preds = model(X_test_tensor).argmax(dim=1).numpy()
        return accuracy_score(y_test, preds)


In [ ]:
 Анализ влияния размерности embedding

In [ ]:
# 🚀 Проверим, как размерность влияет на точность
dims = list(range(1, 33))  # от 1 до 32
scores = [train_model(dim) for dim in dims]

plt.figure(figsize=(10, 5))
sns.lineplot(x=dims, y=scores)
plt.title("📈 Зависимость точности от размерности embedding")
plt.xlabel("Размерность embedding")
plt.ylabel("Accuracy")
plt.grid()
plt.show()


# 📌 Выводы

- Эмбеддинги — это не просто векторы, а логарифмы условных вероятностей в латентном пространстве.
- Байесовский взгляд помогает понять, почему модель "изучает" контексты.
- Слишком малая размерность снижает выразительность.
- Слишком большая — приводит к переобучению и лишним параметрам.
- Размерность embedding стоит подбирать эмпирически, в зависимости от данных.


# 📘 Введение: Позиционные вектора (sinusoidal vs learnable)

Transformer-архитектура **не использует рекурренции или свёртки**, а значит, **не обладает встроенным механизмом понимания порядка**.

👉 Поэтому требуется **позиционное кодирование (positional encoding)**, которое вводит представление о позиции токенов во входной последовательности.

Существуют два основных подхода:
- Синусоидальные позиционн


In [ ]:
# Установка при необходимости
# !pip install torch matplotlib seaborn


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


In [ ]:
# 🔁 Синусоидальное позиционное кодирование (из статьи "Attention is All You Need")
def sinusoidal_positional_encoding(seq_len, dim):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2) * -(np.log(10000.0) / dim))
    
    pe = torch.zeros(seq_len, dim)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


In [ ]:
seq_len = 64
dim = 32

pe = sinusoidal_positional_encoding(seq_len, dim).numpy()

plt.figure(figsize=(12, 6))
for i in range(0, dim, 4):
    plt.plot(pe[:, i], label=f'dim {i}')
plt.title("Синусоидальные позиционные вектора")
plt.xlabel("Position")
plt.ylabel("Value")
plt.legend()
plt.show()


In [ ]:
# Обучаемые позиционные вектора — обычная embedding-матрица
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, seq_len, dim):
        super().__init__()
        self.pe = nn.Parameter(torch.randn(seq_len, dim))

    def forward(self):
        return self.pe


In [ ]:
learnable_pe = LearnablePositionalEncoding(seq_len, dim)().detach().numpy()

plt.figure(figsize=(12, 6))
for i in range(0, dim, 4):
    plt.plot(learnable_pe[:, i], label=f'dim {i}')
plt.title("Обучаемые позиционные вектора")
plt.xlabel("Position")
plt.ylabel("Value")
plt.legend()
plt.show()


# 🧠 Интерпретация

- Синусоидальные вектора фиксированы, они кодируют позиции по определённой схеме:
  - Позволяют модели легко вычислять разности и относительные смещения
  - Устойчивы к extrapolation (длинные последовательности)
- Обучаемые вектора могут адаптироваться к задаче, но не умеют обобщать на длины, не встречавшиеся при обучении

Важно:
- Порядок кодируется через _уникальные шаблоны_, подающиеся на вход вместе с embedding слов
- Эти шаблоны позволяют модели понимать, в каком порядке находятся токены, даже без RNN/CNN


In [ ]:
# Простая модель: сравнение производительности с / без позиционного кодирования
class SimpleTransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, dim, seq_len, use_sinusoidal=True):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.seq_len = seq_len
        self.use_sinusoidal = use_sinusoidal
        
        if use_sinusoidal:
            self.pos_encoding = sinusoidal_positional_encoding(seq_len, dim)
        else:
            self.pos_encoding = nn.Parameter(torch.randn(seq_len, dim))

    def forward(self, x):  # x: (batch_size, seq_len)
        x = self.token_emb(x)  # (batch, seq_len, dim)
        pe = self.pos_encoding if isinstance(self.pos_encoding, torch.Tensor) else self.pos_encoding()
        return x + pe.unsqueeze(0)


In [ ]:
#Демонстрация (продолжение): визуализация суммы токенов и позиций
vocab_size = 100
dim = 16
seq_len = 10
batch = torch.randint(0, vocab_size, (1, seq_len))

model_sin = SimpleTransformerEmbedding(vocab_size, dim, seq_len, use_sinusoidal=True)
model_learn = SimpleTransformerEmbedding(vocab_size, dim, seq_len, use_sinusoidal=False)

out_sin = model_sin(batch).squeeze().detach().numpy()
out_learn = model_learn(batch).squeeze().detach().numpy()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.heatmap(out_sin, cmap="coolwarm")
plt.title("Синусоидальное кодирование")

plt.subplot(1, 2, 2)
sns.heatmap(out_learn, cmap="coolwarm")
plt.title("Обучаемое позиционное кодирование")
plt.tight_layout()
plt.show()


# ✅ Выводы

- Позиционные вектора необходимы для трансформеров, чтобы понять **порядок слов**.
- Синусоидальные:
  - Необучаемые, интерпретируемые
  - Хорошо экстраполируют на длинные последовательности
- Обучаемые:
  - Гибкие, адаптируются к задаче
  - Могут переобучаться на фиксированные длины
- Оба подхода работают, но в разных задачах может быть предпочтение к одному или другому.


# 📘 Введение: Модель Query–Key–Value (QKV)

В основе механизма Self-Attention лежит идея: **какие части входной последовательности важны друг для друга?**

Каждое слово представляется тремя векторами:
- **Query (запрос)** — то, что ищет текущий токен
- **Key (ключ)** — то, что "предлагает" другой токен
- **Value (значение)** — содержимое, которое токен готов передать

Модель Attention вычисляет:
> Насколько хорошо Query одного токена "совпадает" с Key других токенов → это веса внимания → затем применяется к Value

Таким образом модель узнаёт, **на кого обратить внимание при обработке каждого токена**.


In [ ]:
# !pip install torch matplotlib seaborn

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


# 🧮 Формула Scaled Dot-Product Attention

Для входа `X` размерности `(seq_len × d_model)` мы получаем:



In [ ]:
Q = X × W_Q
K = X × W_K
V = X × W_V


Тогда attention считается как:



In [ ]:
Attention(Q, K, V) = softmax(Q × Kᵗ / √d_k) × V


Где:
- `Q × Kᵗ` — сходство между токенами (dot-product)
- `softmax(...)` — веса, куда смотреть
- `× V` — применяем внимание к содержимому

---


In [ ]:
#Генерация случайной последовательности
seq_len = 4
d_model = 8

X = torch.randn(seq_len, d_model)  # вход: 4 токена по 8 признаков
X


In [ ]:
#Построение Q, K, V
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

Q = X @ W_q  # (4×8) × (8×8) = 4×8
K = X @ W_k
V = X @ W_v


In [ ]:
#Внимание (Attention Weights)
d_k = d_model
scores = Q @ K.T / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))  # 4x4
attn_weights = torch.softmax(scores, dim=-1)
attn_weights


In [ ]:
#Итоговое внимание: взвешенное V
out = attn_weights @ V  # (4×4) × (4×8) = 4×8
out


In [ ]:
#Визуализация матрицы внимания
plt.figure(figsize=(6, 4))
sns.heatmap(attn_weights.detach().numpy(), annot=True, cmap="YlGnBu")
plt.title("Матрица внимания (QKᵗ / √d)")
plt.xlabel("Key-позиции")
plt.ylabel("Query-позиции")
plt.show()


In [ ]:
#Объединённый класс Attention (PyTorch)
attn_layer = SelfAttention(d_model)
x_batch = torch.randn(2, seq_len, d_model)  # 2 последовательности

output, weights = attn_layer(x_batch)

print("Выход:")
print(output.shape)  # (2, seq_len, d_model)

print("Внимание:")
print(weights.shape)  # (2, seq_len, seq_len)


# ✅ Выводы

- Механизм QKV позволяет **каждому токену смотреть на другие токены** через вычисление внимания
- Q — описывает текущий запрос
- K — описывает, что представляют другие токены
- V — содержит передаваемую информацию

- Итоговая формула: softmax(Q × Kᵗ / √d) × V
- Это позволяет модели учитывать **взаимосвязи** между всеми словами в предложении

Далее эта идея расширяется на multi-head attention, masking и т.д.


# 🧠 Multi-Head Attention (MHA)

В обычном (single-head) Attention модель "смотрит" на всю последовательность через одну проекцию.  
Но **одного взгляда недостаточно**, чтобы захватить разные аспекты: синтаксис, грамматику, семантику.

### Идея MHA:
- Разделить пространство признаков на части (напр., d_model = 512, num_heads = 8 → каждая голова обрабатывает по 64 признака)
- Независимо применить self-attention для каждой части
- Объединить результаты и снова спроецировать

Таким образом:
> **Multi-Head Attention = несколько "вниманий" параллельно → объединённое представление**


In [ ]:
#Настройки и импорты
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Параметры
seq_len = 6
d_model = 8
num_heads = 2
head_dim = d_model // num_heads


# 🔬 Основная формула

MHA = Concat(head₁, ..., headₙ) × Wᵒ

Где каждая headᵢ:


In [ ]:
Attention(Qᵢ, Kᵢ, Vᵢ) = softmax(Qᵢ × Kᵢᵗ / √dₕ) × Vᵢ

Здесь `Qᵢ = X × W_Qᵢ`, `Kᵢ = X × W_Kᵢ`, `Vᵢ = X × W_Vᵢ`  
Каждая "голова" имеет свои W_Q, W_K, W_V  
Затем объединяются через Wᵒ


In [ ]:
#Пример вручную (один батч, 2 головы)
x = torch.randn(seq_len, d_model)

# Общие проекции (упрощение: вручную разбиваем)
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)
W_o = torch.randn(d_model, d_model)

Q = x @ W_q
K = x @ W_k
V = x @ W_v

# Разбиваем на головы (reshape)
Q_heads = Q.reshape(seq_len, num_heads, head_dim).transpose(0, 1)  # [heads, seq_len, head_dim]
K_heads = K.reshape(seq_len, num_heads, head_dim).transpose(0, 1)
V_heads = V.reshape(seq_len, num_heads, head_dim).transpose(0, 1)

# Attention по каждой голове
scores = Q_heads @ K_heads.transpose(1, 2) / np.sqrt(head_dim)  # [heads, seq, seq]
weights = torch.softmax(scores, dim=-1)
attn_out = weights @ V_heads  # [heads, seq_len, head_dim]

# Объединение голов обратно
attn_out = attn_out.transpose(0, 1).reshape(seq_len, d_model)
output = attn_out @ W_o  # Финальный linear layer
output


In [ ]:
 #Класс MultiHeadAttention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model должно делиться на количество голов"
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, d_model * 3)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape  # Batch, Time, Dim
        qkv = self.qkv_proj(x)  # (B, T, 3 * D)
        qkv = qkv.reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]  # [B, heads, T, head_dim]

        scores = Q @ K.transpose(-2, -1) / np.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = attn @ V  # [B, heads, T, head_dim]

        out = out.transpose(1, 2).reshape(B, T, D)
        return self.out_proj(out), attn


In [ ]:
#Пример на батче
mha = MultiHeadAttention(d_model=8, num_heads=2)
x = torch.randn(1, 6, 8)  # batch_size=1, seq_len=6, d_model=8

output, attn = mha(x)

print("Output shape:", output.shape)   # [1, 6, 8]
print("Attention shape:", attn.shape)  # [1, 2, 6, 6]


In [ ]:
#Визуализация внимания для каждой головы
fig, axes = plt.subplots(1, num_heads, figsize=(12, 4))
for i in range(num_heads):
    sns.heatmap(attn[0, i].detach().numpy(), cmap="YlGnBu", ax=axes[i], cbar=False)
    axes[i].set_title(f"Голова {i+1}")
    axes[i].set_xlabel("Key")
    axes[i].set_ylabel("Query")
plt.suptitle("Веса внимания в каждой голове")
plt.show()


# ✅ Выводы

- Multi-Head Attention — мощное расширение обычного внимания
- Каждая голова "смотрит" под своим углом
- Это даёт модели возможность параллельно учитывать:
  - близость слов
  - синтаксические связи
  - длинные зависимости

➡️ В отличие от single-head, MHA лучше обучается и устойчивее к шуму

Следующим шагом идёт **Position-wise Feedforward Layer**, а также **LayerNorm** и **Residual Connections**
